# 02 · Attention Weight Visualization

- **Objectives**: enable per-layer attention capture, extract the `batch × heads × seq × seq` weight tensors, and plot them as heatmaps.
- **Requirements**: 1 GPU (or CPU — works but slower).
- **Runtime**: ~15 seconds.

## Setup

In [ ]:
import matplotlib.pyplot as plt
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Build a small model

We use a narrow model so the heatmaps are readable. `n_layers=4 × n_heads=4` gives us a 4×4 grid of attention patterns to inspect.

In [ ]:
from kempnerforge.config.schema import ModelConfig
from kempnerforge.model.transformer import Transformer

config = ModelConfig(
    dim=128,
    n_layers=4,
    n_heads=4,
    n_kv_heads=4,
    vocab_size=256,
    max_seq_len=64,
)

model = Transformer(config).to(device)
model.eval()
print(f"{config.n_layers} layers × {config.n_heads} heads")

## Enable attention weight capture

Setting `capture_attention_weights = True` on each `Attention` module makes it compute attention manually (`softmax(QK^T / √d)`) and stash the weights on CPU after each forward pass. This disables the fused SDPA kernel, so it's significantly slower — only use it for analysis, never training.

In [ ]:
for layer in model.layers.values():
    layer.attention.capture_attention_weights = True

## Forward pass

Deterministic token sequence so the visualization is reproducible across runs.

In [ ]:
torch.manual_seed(0)
seq_len = 32
tokens = torch.randint(0, config.vocab_size, (1, seq_len), device=device)

with torch.no_grad():
    _ = model(tokens)

print(f"Forward pass complete ({seq_len} tokens).")

## Collect attention weights per layer

Each layer stores its last attention tensor in `last_attention_weights` with shape `(batch, n_heads, seq_q, seq_k)` on CPU.

In [ ]:
attn_weights = {}
for name, layer in model.layers.items():
    w = layer.attention.last_attention_weights  # (B, H, Sq, Sk), on CPU
    if w is not None:
        attn_weights[int(name)] = w[0]  # drop batch dim → (H, Sq, Sk)

for layer_idx, w in attn_weights.items():
    print(f"Layer {layer_idx}: shape={tuple(w.shape)}, sum/row≈{w[0].sum(-1).mean().item():.3f}")

Row sums are all ≈1.0 as expected (softmax over keys).

## Plot a layer × head grid

Each subplot is a single attention head's `(seq_q, seq_k)` matrix. The lower-triangular structure is the causal mask — queries can only attend to past keys.

In [ ]:
n_layers = len(attn_weights)
n_heads = config.n_heads

fig, axes = plt.subplots(n_layers, n_heads, figsize=(2.2 * n_heads, 2.2 * n_layers))
for layer_idx, w in attn_weights.items():  # w: (H, Sq, Sk)
    for head in range(n_heads):
        ax = axes[layer_idx][head]
        ax.imshow(w[head].numpy(), cmap="viridis", aspect="auto")
        ax.set_xticks([])
        ax.set_yticks([])
        if head == 0:
            ax.set_ylabel(f"Layer {layer_idx}")
        if layer_idx == 0:
            ax.set_title(f"Head {head}")

plt.suptitle("Attention weights — random-init model, random tokens", y=1.01)
plt.tight_layout()
plt.show()

## Interpretation

- The **lower-triangular** structure is the causal mask: queries only attend to past keys.
- With a **random-initialized** model and **random tokens**, patterns are mostly uniform within the causal region — there's no learned structure yet.
- After training, heads develop specialized patterns (induction heads, positional heads, name-mover heads, etc.). Try this notebook again against a checkpoint you've trained (see [`04_checkpoint_analysis.ipynb`](04_checkpoint_analysis.ipynb)).

## Clean up

Disable capture when you're done so subsequent forward passes use the fused (fast) SDPA kernel again.

In [ ]:
for layer in model.layers.values():
    layer.attention.capture_attention_weights = False